In [1]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [54]:
!airflow version

3.0.0


In [3]:
import os
os.environ['AIRFLOW_HOME'] = '/root/airflow'

!pip install "apache-airflow==3.0.0" \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-3.0.0/constraints-3.12.txt" \
  --no-cache-dir -q

print("Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.0/152.0 kB 55.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.2/65.2 kB 227.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.2/233.2 kB 315.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 238.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 326.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 266.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 289.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 207.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 235.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 kB 155.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 216.4 MB/s eta 0:00

In [4]:
!airflow version

3.0.0


In [5]:
!airflow db migrate

[2026-05-03T18:37:12.648+0000] {providers_manager.py:946} INFO - The hook_class 'airflow.providers.standard.hooks.filesystem.FSHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
[2026-05-03T18:37:12.651+0000] {providers_manager.py:946} INFO - The hook_class 'airflow.providers.standard.hooks.package_index.PackageIndexHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
DB: sqlite:////root/airflow/airflow.db
Performing upgrade to the metadata database sqlite:////root/airflow/airflow.db
[2026-05-03T18:37:13.026+0000] {migration.py:207} INFO - Context impl SQLiteImpl.
[2026-05-03T18:37:13.026+0000] {migration.py:210} INFO - Will assume non-transactional DDL.
[2026-05-03T18:37:13.027+0000] {migration.py:207} INFO - Context impl SQLiteImpl.
[2026-0

# Connecting to Airflow UI

In [45]:
import os, subprocess, time

!pkill -f "airflow" 2>/dev/null || true
!pkill -f "gunicorn" 2>/dev/null || true
time.sleep(3)

os.environ['AIRFLOW_HOME'] = '/root/airflow'

standalone = subprocess.Popen(
    ['airflow', 'standalone'],
    env={**os.environ, 'AIRFLOW__API__PORT': '8081'},
    stdout=open('/root/airflow/standalone.log', 'w'),
    stderr=subprocess.STDOUT
)

print("Waiting 45 seconds...")
time.sleep(45)
print("Done!")

^C
^C
Waiting 45 seconds...
Done!


In [8]:
!pip install pyngrok -q

In [46]:
!ss -tlnp | grep LISTEN

LISTEN 0      4096      127.0.0.11:40501      0.0.0.0:*                                                                                                                           
LISTEN 0      4096       127.0.0.1:4040       0.0.0.0:*    users:(("ngrok",pid=6493,fd=4))                                                                                        
LISTEN 0      128        127.0.0.1:3453       0.0.0.0:*    users:(("colab-fileshim.",pid=70,fd=3))                                                                                
LISTEN 0      100        127.0.0.1:44429      0.0.0.0:*    users:(("python3",pid=2616,fd=28))                                                                                     
LISTEN 0      128      172.28.0.12:9000       0.0.0.0:*    users:(("jupyter-server",pid=91,fd=7))                                                                                 
LISTEN 0      2048         0.0.0.0:8081       0.0.0.0:*    users:(("python3",pid=8818,fd=3),("python3",pi

Kill if exisitng tunnel is Open for ngrok

In [50]:
!pkill -f ngrok 2>/dev/null || true

import time
time.sleep(3)

# Verify ngrok is dead
!ss -tlnp | grep 4040 || echo "✅ ngrok fully stopped"

^C
✅ ngrok fully stopped


In [51]:
from pyngrok import ngrok, conf

# Full reset
conf.get_default().auth_token = "3DDfZi2LFOCnrH8XLD9cHcGZvrG_3MJQoCueUp8RuCJkbf8o5"

# Connect once only
public_url = ngrok.connect(8081)
print(f"✅ Open this: {public_url}")

✅ Open this: NgrokTunnel: "https://smuggling-either-bunkbed.ngrok-free.dev" -> "http://localhost:8081"


In [59]:
!ss -tlnp | grep LISTEN

LISTEN 0      4096      127.0.0.11:40501      0.0.0.0:*                                                                                                                           
LISTEN 0      4096       127.0.0.1:4040       0.0.0.0:*    users:(("ngrok",pid=10339,fd=4))                                                                                       
LISTEN 0      128        127.0.0.1:3453       0.0.0.0:*    users:(("colab-fileshim.",pid=70,fd=3))                                                                                
LISTEN 0      100        127.0.0.1:44429      0.0.0.0:*    users:(("python3",pid=2616,fd=28))                                                                                     
LISTEN 0      128      172.28.0.12:9000       0.0.0.0:*    users:(("jupyter-server",pid=91,fd=7))                                                                                 
LISTEN 0      2048         0.0.0.0:8081       0.0.0.0:*    users:(("python3",pid=8818,fd=3),("python3",pi

Get user Name and Password

In [65]:
!cat /root/airflow/simple_auth_manager_passwords.json.generated

{"admin": "kcNKMt4XMUbQYVMV"}


# DAG Creation

In [76]:
dag_code = """
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def say_hello():
    print("Hello from Airflow! My first DAG is working!")

def say_goodbye():
    print("Goodbye! DAG completed successfully.")

with DAG(
    dag_id='my_first_dag',
    start_date=datetime(2024, 1, 1),
    schedule='@daily',
    catchup=False,
    tags=['beginner']
) as dag:

    task1 = PythonOperator(
        task_id='say_hello',
        python_callable=say_hello
    )

    task2 = PythonOperator(
        task_id='say_goodbye',
        python_callable=say_goodbye
    )

    task1 >> task2
"""

with open('/root/airflow/dags/my_first_dag.py', 'w') as f:
    f.write(dag_code)

print("✅ DAG updated!")

✅ DAG updated!


In [63]:
!ls -la /root/airflow/dags/

total 12
drwxr-xr-x 2 root root 4096 May  3 19:07 .
drwxr-xr-x 4 root root 4096 May  3 19:12 ..
-rw-r--r-- 1 root root  631 May  3 19:12 my_first_dag.py


In [77]:
!cat /root/airflow/dags/my_first_dag.py


from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

def say_hello():
    print("Hello from Airflow! My first DAG is working!")

def say_goodbye():
    print("Goodbye! DAG completed successfully.")

with DAG(
    dag_id='my_first_dag',
    start_date=datetime(2024, 1, 1),
    schedule='@daily',
    catchup=False,
    tags=['beginner']
) as dag:

    task1 = PythonOperator(
        task_id='say_hello',
        python_callable=say_hello
    )

    task2 = PythonOperator(
        task_id='say_goodbye',
        python_callable=say_goodbye
    )

    task1 >> task2


 Check where Airflow is looking for DAGs

In [71]:
!airflow config get-value core dags_folder
!ls /root/airflow/

[2026-05-03T19:24:47.496+0000] {providers_manager.py:946} INFO - The hook_class 'airflow.providers.standard.hooks.filesystem.FSHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
[2026-05-03T19:24:47.500+0000] {providers_manager.py:946} INFO - The hook_class 'airflow.providers.standard.hooks.package_index.PackageIndexHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
/root/airflow/dags
airflow.cfg  airflow.db-shm  dags  simple_auth_manager_passwords.json.generated
airflow.db   airflow.db-wal  logs  standalone.log


In [72]:
!cat /root/airflow/airflow.cfg | grep -i "bundle\|dags_folder\|dag_folder"

# Variable: AIRFLOW__CORE__DAGS_FOLDER
dags_folder = /root/airflow/dags
#   - DAG bundles, which allows Airflow to load DAGs from different sources
# String path to folder where Airflow bundles can store files locally. Not templated.
# Example: dag_bundle_storage_path = /tmp/some-place
# Variable: AIRFLOW__DAG_PROCESSOR__DAG_BUNDLE_STORAGE_PATH
# dag_bundle_storage_path = 
# The default is the dags folder dag bundle.
# Example: dag_bundle_config_list = [
#         "classpath": "airflow.providers.git.bundles.git.GitDagBundle",
# Variable: AIRFLOW__DAG_PROCESSOR__DAG_BUNDLE_CONFIG_LIST
dag_bundle_config_list = [
        "classpath": "airflow.dag_processing.bundles.local.LocalDagBundle",
# How often (in seconds) to refresh, or look for new files, in a DAG bundle.
# Always run tasks with the latest code.  If set to True, the bundle version will not
# Variable: AIRFLOW__DAG_PROCESSOR__DISABLE_BUNDLE_VERSIONING
disable_bundle_versioning = False
# How often the DAG processor should check if a

In [78]:
!airflow dags reserialize

[2026-05-03T19:30:35.586+0000] {providers_manager.py:946} INFO - The hook_class 'airflow.providers.standard.hooks.filesystem.FSHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
[2026-05-03T19:30:35.597+0000] {providers_manager.py:946} INFO - The hook_class 'airflow.providers.standard.hooks.package_index.PackageIndexHook' is not fully initialized (UI widgets will be missing), because the 'flask_appbuilder' package is not installed, however it is not required for Airflow components to work
[2026-05-03T19:30:36.498+0000] {manager.py:120} INFO - DAG bundles loaded: dags-folder, example_dags
[2026-05-03T19:30:37.099+0000] {dagbag.py:575} INFO - Filling up the DagBag from /root/airflow/dags
[2026-05-03T19:30:37.149+0000] {dag.py:1894} INFO - Sync 1 DAGs
[2026-05-03T19:30:37.189+0000] {dag.py:2517} INFO - Setting next_dagrun for my_first_dag to 2026-05-03 00:00:00+00:0

In [79]:
import time
time.sleep(15)
!airflow dags list 2>/dev/null | grep -i "my_first\|dag_id"

dag_id                                                  | fileloc                                                                                  | owners  | is_paused | bundle_name  | bundle_version
my_first_dag                                            | /root/airflow/dags/my_first_dag.py                                                       | airflow | True      | dags-folder  | None          


View the List of DAGS

In [70]:
import time
print("Waiting 20 seconds for dag-processor to pick it up...")
time.sleep(20)
!airflow dags list

Waiting 20 seconds for dag-processor to pick it up...
              |              |         |           |              | bundle_versi
dag_id        | fileloc      | owners  | is_paused | bundle_name  | on          
==============+==============+=========+===========+==============+=============
asset1_produc | /usr/local/l | airflow | True      | example_dags | None        
er            | ib/python3.1 |         |           |              |             
              | 2/dist-packa |         |           |              |             
              | ges/airflow/ |         |           |              |             
              | example_dags |         |           |              |             
              | /example_ass |         |           |              |             
              | et_decorator |         |           |              |             
              | .py          |         |           |              |             
asset2_produc | /usr/local/l | airflow | True      | ex